# Healpix

Healpix is a modern pixelisation algorithm used for dividing the sky into equal area cells (see https://en.wikipedia.org/wiki/HEALPix or https://arxiv.org/abs/astro-ph/0409513 for more)

The number of pixels over the sky is characterised by the NSIDE parameter, where $N_{pixels} = 12\times\text{NSIDE}^2$

Within python the primary package for using HealPix is called HealPy, which allows 1D $N_{pixels}$ arrays to treated as healpix maps

In [ ]:
import healpy as hp
import numpy as np

for nside in [1,2,4,16]:
    npix = hp.nside2npix(nside)
    
    m = np.arange(npix)
    
    hp.visufunc.mollview(m)

Healpix is a very handy and increasingly common way to store survey data that corresponds to a significant portion of the total sky. As a result downloading healpix maps and visualising survey data across the sky is easy! Lets try one

In [ ]:
!curl -O https://faun.rc.fas.harvard.edu/dfink/skymaps/halpha/data/v1_1/healpix/Halpha_fwhm06_0512.fits

As a FITS file, this data also has a header full of useful information about the data products

In [ ]:
from astropy.io import fits
import numpy as np
halpha_map = hp.fitsfunc.read_map('Halpha_fwhm06_0512.fits')
halpha_hdr = fits.getheader('Halpha_fwhm06_0512.fits')

As can be seen below the data we are using has an NSIDE of 512, resulting in 3145728 pixels, with each pixel measuring flux of H$\alpha$ at that position in Rayleighs

In [ ]:
# look at the header
# print the keys of the header

So how does the data look? We can use a mollweide projection from healpy to show this data in 2D (https://en.wikipedia.org/wiki/Mollweide_projection)

In [ ]:
hp.visufunc.mollview(halpha_map)

Strangely this map should look pretty empty with only a few bright points. This is a classic trap, try depicting in log space instead

In [ ]:
hp.visufunc.mollview(#show the log of the map)

Ahh, much better we can see now that there is indeed plenty of ther Halpha emission across the Galactic plane, but the strength of the emission varies over many orders of magnitude and so it becomes washed out if we use a linear scale. 

If we really want to visualise this on a linear scale, we can mask some of the intense regions to allow us to visualise the low intensities better. 

In [ ]:
halpha_mask = np.log10(halpha_map)<#find an appropriate masking value

hp.visufunc.mollview((halpha_map)#apply the mask

Helpfully Healpix can make use of coordinate aware datasets, in this case we know that the data is in the Galactic coordinate system (places the Galactic plane along the equator), this allows us to simply transform the data to an equatorial coordinate system (RA,Dec).

In [ ]:
hp.visufunc.mollview((halpha_map)*halpha_mask, coord=['G','C'])

We might also be more interested in looking at the edges of the above map, Healpy allows us to easily rotate the coordinates and centre a particular part of the map using the `rot=` variable in mollview. This can be a great way to build intuition about how Galactic sources transit overhead

In [ ]:
hp.visufunc.mollview((halpha_map)*halpha_mask, coord=['G','C'], rot=#what should you rotate by

# Bilby 

Bilby (Bayesian Inference Library) is a bespoke fitting package that was designed for analysing gravitational wave signatures. Following that development is has become a popular way to deploy bayesian fitting routines like MCMC or Nested Sampling.

We demonstrate how to use it here to recover the input parameters we use to simulate a fake burst profile.

First lets create a gaussian pulse and add some noise to it

In [ ]:
from scipy.signal.windows import gaussian
import matplotlib.pyplot as plt
import bilby
def fakeProfile(width, amplitude):
    t = np.random.normal(size=100)
    t = t +gaussian(100, width)*amplitude
    return t

In [ ]:
tProf = fakeProfile(5,10)
plt.plot(tProf)

Then lets define a function we want to fit to this data. In our case this is simple because we can use the same gaussian profile but without the noise added

In [ ]:
def pulseFit(time, w1, a1):
    model = #what should the model be from the above fake profile function
    return model

Next we need to define some priors, these are basically our initial guesses for how likely a particular solution might be. Lets assume we don't know anything about our fake pulse, and so the natural chocie is assume any properties are equally likely within some reasonable boundaries, this is known as using a uniform or uninformative prior and is very common.

In [ ]:
priors=bilby.core.prior.PriorDict()
priors["w1"]=bilby.core.prior.Uniform(name='$w$', minimum=#, maximum=#) pick some good priors
priors["a1"]=bilby.core.prior.Uniform(name='a', minimum=#, maximum=#)
priors["sigma"]=bilby.core.prior.LogUniform(name='error',minimum=#,maximum=#)

Then we need to define a likelihood function that defines how likely differences between the data and the fit model are. A common choice here is use a gaussian likelihood (basically assuming that you're uncertainty on the data points is normally distributed, which is is here as we used noise that was gaussian)

then we can feed in what we are trying to fit, the x and y values of our profile as well as the model function we want to fit. The model function must have a specific structure that match f(x, prior arguments..)

In [ ]:
likelihood = bilby.core.likelihood.GaussianLikelihood(x=np.arange(len(tProf)), y=tProf,func=pulseFit, sigma=None)

Then we can do the fitting using whichever bayesian method we prefer (here we use dynesty, as nested sampling algorithm)

In [ ]:
result1 = bilby.sampler.run_sampler(likelihood=likelihood, priors=priors,sampler='dynesty',nlive=500,clean=True,outdir='.',label='PulseFit',verbose=True,resume=False)


Once the above fitting is complete (should take a ~minute) we can display the results in a corner plot that shows us how the fit parameters are correlated, the best fit answers and the associated uncertainties

In [ ]:
result1.plot_corner(parameters=['w1','a1','sigma'], labels=['$w$', '$a$', 'error'], priors=None,filename='pulseFit.pdf',save=True)


# Astropy, units and cosmology


A helpful package for unit conversion in any scientific analysis is astropy.

The units routines allow values in python to be assigned measurement units, and provides easy ways to convert between them.

In [ ]:
from astropy import units as u

Lets try something simple like transforming velocities, to establish a velocity we simply multiply by the relevant units

In [ ]:
v = 10 * u.m/u.s

This velocity value can then be converted to different units using the `.to()` method

In [ ]:
v.to(u.km/u.hr)

We can also use it for dimensional analysis of more complex equations such as the gravitational collapse timescale

In [ ]:
from astropy.constants import G

R = 1 #multiply by a length unit
M = 1 #multiply by a mass unit

t = (np.pi**2*(R**3) / (8 * G * M))**0.5

print(t)
print(t.decompose())

For those interested in cosmology, astropy also provides a great way to calculate distances for a given expanding universe. For example lets use the results from the latest Planck survey results

In [ ]:
from astropy.cosmology import Planck18 as cosmo

with this cosmology imported we can simply calculate complicated quantities like the difference between luminosity, angular diameter and comoving distances out to a particular redshift

In [ ]:
z = np.arange(0,#pick a maximum redshift,0.01)
plt.plot(z, cosmo.comoving_distance(z),label='Comoving')
plt.plot(z, cosmo.angular_diameter_distance(z), label='Angular Diameter')
plt.plot(z, cosmo.luminosity_distance(z), label='luminosity')
plt.xlabel('Redshift')
plt.ylabel('Distance (Mpc)')
plt.legend()

# Using colour sets to make images


Default functionality for python also allows you to make heatmaps with red green and blue channels which are common to most images, allowing for realistic image plotting. Lets try this out with some astronomy images

In [ ]:
from astroquery.skyview import SkyView
from astroquery.mast import Observations
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy import wcs

Most of the above packages are helpful for astronomy but you don't need to know too much about them specifically, you can simply use the below function to download image data for a particular line of sight

In [ ]:
def visualiseLocation(ra,dec,ID):

    pos = SkyCoord(
        ra=ra * u.deg,
        dec=dec * u.deg,
        frame='icrs'
    )
    images = SkyView.get_images(
        position=pos,
        survey=['DSS2 Blue', 'DSS2 Red', 'DSS2 IR'],
        radius=0.5 * u.deg,
        coordinates='J2000'
    )

    # Save them
    cols = ['g','r','i']
    for i, hdu_list in enumerate(images):
        hdu_list.writeto(str(ID)+'_dss2_'+cols[i]+'.fits', overwrite=True)

Put in an RA and Dec of your choice as well as some unique ID you want to write the file too

In [ ]:
visualiseLocation(#pick a sky location)

Once download (~few minutes) we can load in this data and use the different frequencies as our red green and blue channels as is common for false colour images in astronomy

In [ ]:
ID=#replace with your id
hdr = fits.getheader(f"{ID}_dss2_g.fits")
proj = wcs.WCS(hdr)

g = fits.getdata(f"{ID}_dss2_g.fits").astype(float)
r = fits.getdata(f"{ID}_dss2_r.fits").astype(float)
i = fits.getdata(f"{ID}_dss2_i.fits").astype(float)




Each of the g, r, i filters is a different 2D array of intensities at different sky positions, we can visualise these simply using matplotlib's imshow function

In [ ]:
plt.imshow(g)
plt.show()

Furthermore if we normalise each of those arrays to between zero and one, we can then concatenate them into one big array (x,y,3) and feed that to imshow instead and it will plot a false colour image using our different frequencies

In [ ]:
plt.imshow(np.concatenate([g[:,:,np.newaxis]/np.amax(g),r[:,:,np.newaxis]/np.amax(r),i[:,:,np.newaxis]/np.amax(i)],axis=2))
plt.show()

Now for the most appropriate normalisations we want to use some standard image processing techniques and use the header information of each of these fits files to actually plot the sky position of the data 

In [ ]:
from astropy.stats import sigma_clip
from astropy.visualization import PercentileInterval, AsinhStretch


def make_asinh_rgb(g, r, i,
                   percentile=99.95,
                   sigma=3,
                   weights=(1/3, 1/3, 1/3)):
    """
    Create an RGB image from SDSS gri bands using shared percentile
    scaling and an asinh stretch.

    Parameters
    ----------
    g, r, i : 2D numpy arrays
        Background-included images.
    percentile : float
        Upper percentile for intensity clipping (e.g. 99.5).
    sigma : float
        Sigma for background subtraction clipping.
    weights : tuple
        Weights for combined intensity (wg, wr, wi).

    Returns
    -------
    rgb : 3D numpy array
        Stretched RGB image scaled to [0,1].
    """

    def subtract_bg(img):
        clipped = sigma_clip(img, sigma=sigma)
        return img - np.median(clipped)

    # Background subtraction
    g = subtract_bg(g)
    r = subtract_bg(r)
    i = subtract_bg(i)

    # Combined intensity
    wg, wr, wi = weights
    I = wg * g + wr * r + wi * i

    # Shared percentile limits
    interval = PercentileInterval(percentile)
    vmin, vmax = interval.get_limits(I)

    stretch = AsinhStretch()

    def scale_band(img):
        scaled = np.clip((img - vmin) / (vmax - vmin), 0, 1)
        return stretch(scaled)

    R = scale_band(i)
    G = scale_band(r)
    B = scale_band(g)

    rgb = np.dstack([R, G, B])

    return np.clip(rgb, 0, 1)

In [ ]:
rgb = make_asinh_rgb(g,r,i)

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection=proj)

ax.imshow(rgb, origin="lower")